# Potamogaton XAI 
In our data, we realised there are about 105 images across separate species of the same genus - potamogaton. That is alomost half of our dataset.

## Objective of this notebook
Our objective is to test what features the models focus on to make a distinction between them and compare these to the ones mentioned in botanical textbooks.

We will use Explainable AI (XAI) techniques to do this. One of the most common XAI techniques is Grad-CAM, which produces "heatmaps" that highlight the areas of an image that are most important for a model's prediction.

In [1]:
PROJECT_ROOT = "/home/takayuki/Desktop/summer2025/plants" 
DATASET_DIR = "/home/takayuki/Desktop/summer2025/plants/data/AquaticPlantLabData/potamogaton"

# 1. DataPrep

In [2]:
random_state = 42

batch_size = 4

train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

In [ ]:
import os
import torch
import torch.nn as nn  
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
import numpy as np
from collections import Counter



# --- Transforms ---
train_tf = v2.Compose([
    v2.Resize((224, 224)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=20),
    v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    v2.ToTensor(),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

basic_tf = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToTensor(),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# --- Dataset and Splitting ---

# Load the full dataset once to get all info
full_dataset = ImageFolder(root=DATASET_DIR)

class_names = full_dataset.classes
class_to_idx = full_dataset.class_to_idx
num_classes = len(class_names)

# Perform the stratified split on indices
indices = list(range(len(full_dataset)))
labels = full_dataset.targets

train_val_indices, test_indices = train_test_split(
    indices,
    test_size=test_ratio,
    stratify=labels,
    random_state=random_state
)

train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=val_ratio / (train_ratio + val_ratio),
    stratify=[labels[i] for i in train_val_indices],
    random_state=random_state
)

class DatasetWrapper(Dataset):
    """A wrapper to apply specific transforms to a subset of a dataset."""
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.indices)

train_dataset = DatasetWrapper(full_dataset, train_indices, transform=train_tf)
val_dataset = DatasetWrapper(full_dataset, val_indices, transform=basic_tf)
test_dataset = DatasetWrapper(full_dataset, test_indices, transform=basic_tf)


# Create DataLoaders
train_dl = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_dl = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_dl = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print("DataLoaders created successfully.")

# --- Verification ---
train_labels = [labels[i] for i in train_indices]
val_labels = [labels[i] for i in val_indices]
test_labels = [labels[i] for i in test_indices]

train_class_counts = Counter(train_labels)
val_class_counts = Counter(val_labels)
test_class_counts = Counter(test_labels)

# Print class distribution
print("-" * 50)
print(f"Found {num_classes} classes. Distribution across splits:")
print("-" * 50)
print(f"{'Class Name':<25} {'Train':>7} {'Val':>7} {'Test':>7}")
print("-" * 50)
for idx, class_name in enumerate(class_names):
    print(f"{class_name:<25} {train_class_counts.get(idx, 0):>7} {val_class_counts.get(idx, 0):>7} {test_class_counts.get(idx, 0):>7}")
print("-" * 50)
print(f"{'Total':<25} {len(train_dataset):>7} {len(val_dataset):>7} {len(test_dataset):>7}")
print("-" * 50)

DataLoaders created successfully.
--------------------------------------------------
Found 7 classes. Distribution across splits:
--------------------------------------------------
Class Name                  Train     Val    Test
--------------------------------------------------
Potamogeton crispus            17       2       2
Potamogeton gramineus          13       2       2
Potamogeton illinoensis        28       3       4
Potamogeton natans              7       1       1
Potamogeton praelongus          6       1       1
Potamogeton richardsonii        5       1       0
Potamogeton robbinsii           7       1       1
--------------------------------------------------
Total                          83      11      11
--------------------------------------------------


/home/takayuki/.local/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


# Model Training

In [4]:
import timm
from torchinfo import summary

timm.list_models("convnextv2*", pretrained=True)

['convnextv2_atto.fcmae',
 'convnextv2_atto.fcmae_ft_in1k',
 'convnextv2_base.fcmae',
 'convnextv2_base.fcmae_ft_in1k',
 'convnextv2_base.fcmae_ft_in22k_in1k',
 'convnextv2_base.fcmae_ft_in22k_in1k_384',
 'convnextv2_femto.fcmae',
 'convnextv2_femto.fcmae_ft_in1k',
 'convnextv2_huge.fcmae',
 'convnextv2_huge.fcmae_ft_in1k',
 'convnextv2_huge.fcmae_ft_in22k_in1k_384',
 'convnextv2_huge.fcmae_ft_in22k_in1k_512',
 'convnextv2_large.fcmae',
 'convnextv2_large.fcmae_ft_in1k',
 'convnextv2_large.fcmae_ft_in22k_in1k',
 'convnextv2_large.fcmae_ft_in22k_in1k_384',
 'convnextv2_nano.fcmae',
 'convnextv2_nano.fcmae_ft_in1k',
 'convnextv2_nano.fcmae_ft_in22k_in1k',
 'convnextv2_nano.fcmae_ft_in22k_in1k_384',
 'convnextv2_pico.fcmae',
 'convnextv2_pico.fcmae_ft_in1k',
 'convnextv2_tiny.fcmae',
 'convnextv2_tiny.fcmae_ft_in1k',
 'convnextv2_tiny.fcmae_ft_in22k_in1k',
 'convnextv2_tiny.fcmae_ft_in22k_in1k_384']

In [5]:
timm_model_name = "convnextv2_nano.fcmae_ft_in22k_in1k"
model = timm.create_model(timm_model_name, pretrained=True, num_classes=num_classes)
summary(model, input_size=(batch_size, 3, 224, 224))

Layer (type:depth-idx)                                  Output Shape              Param #
ConvNeXt                                                [4, 7]                    --
├─Sequential: 1-1                                       [4, 80, 56, 56]           --
│    └─Conv2d: 2-1                                      [4, 80, 56, 56]           3,920
│    └─LayerNorm2d: 2-2                                 [4, 80, 56, 56]           160
├─Sequential: 1-2                                       [4, 640, 7, 7]            --
│    └─ConvNeXtStage: 2-3                               [4, 80, 56, 56]           --
│    │    └─Identity: 3-1                               [4, 80, 56, 56]           --
│    │    └─Sequential: 3-2                             [4, 80, 56, 56]           112,800
│    └─ConvNeXtStage: 2-4                               [4, 160, 28, 28]          --
│    │    └─Sequential: 3-3                             [4, 160, 28, 28]          51,520
│    │    └─Sequential: 3-4                    

# Traning

In [6]:
TRAINING_BASE_DIR = os.path.join(PROJECT_ROOT, "training/potamogaton") 
checkpoint_path = os.path.join(TRAINING_BASE_DIR, "checkpoints")
models_path = os.path.join(TRAINING_BASE_DIR, "models")
base_logs_path = os.path.join(TRAINING_BASE_DIR, "runs/convnextv2")

os.makedirs(checkpoint_path, exist_ok=True)
os.makedirs(models_path, exist_ok=True)
os.makedirs(base_logs_path, exist_ok=True)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device != torch.device("cuda"):
    raise RuntimeError("CUDA is not available. It is preffered to run this script on a GPU for training.")
    # You can comment out this line if you want to run on CPU for testing purposes.
    
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.to(device)
print(f"Using device: {device}, {torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU'}")

Using device: cuda, NVIDIA RTX A500 Laptop GPU


In [8]:
from datetime import datetime
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter

training_run_name = "g1_pot"
num_epochs = 50

label_smoothing = 0.05  

weight_decay = 2e-5

lr_initial = 0.0005
lr_cosine_minimum = 0.00002
lr_cosine_tmax_epochs = num_epochs

# checkpoint_interval = 100
validation_interval = 1

# Early stopping parameters ---------------------------------------------------
# We stop when both loss and accuracy do not improve for a certain number of epochs, respectively
early_stopping = False
patience_loss = 30        
patience_accuracy = 30  


2025-09-03 10:30:52.108197: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-03 10:30:52.130022: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756875652.153373   46721 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756875652.160186   46721 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756875652.178806   46721 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [9]:
criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))
optimizer = torch.optim.AdamW(trainable_params, lr=lr_initial, weight_decay=weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=lr_cosine_tmax_epochs, eta_min=lr_cosine_minimum)

now = datetime.now()
timestamp = now.strftime("%m-%d_%H-%M--%S")


hparams_dict = {
    "a_run_name": training_run_name,
    "data_directory": DATASET_DIR,
    "train_dataset_percent": train_ratio,
    "val_dataset_percent": val_ratio,
    "test_dataset_percent": test_ratio,
    "model_architecture": timm_model_name,
    "num_trainable_parameters": sum(p.numel() for p in trainable_params),
    "optimizer": optimizer.__class__.__name__,
    "criterion": criterion.__class__.__name__,
    "scheduler": scheduler.__class__.__name__,
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "initial_learning_rate": lr_initial,
    "final_learning_rate": lr_cosine_minimum,
    "weight_decay": weight_decay,
    "random_seed": random_state,
    "timestamp": timestamp
}

run_id = f"{timestamp}_{training_run_name}_{timm_model_name}"

writer = SummaryWriter(log_dir=os.path.join(base_logs_path, run_id))

for key, value in hparams_dict.items():
    writer.add_text(key, str(value))

completed_all_epochs = False

In [10]:
def save_model_weights(model, path, verbose=False):
    torch.save(model.state_dict(), path)
    if verbose:
        print(f"Model weights saved to {path}")

In [ ]:
import time
from tqdm import tqdm

# Training loop
best_val_loss = float('inf') 
best_loss_epoch = 0
best_val_accuracy = 0.0
best_acc_epoch = 0

patience_counter_loss = 0
patience_counter_acc = 0

print("="*100)
print(f"Starting training for run: {run_id}")
print("="*100)

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_dl, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / total
    train_accuracy = correct / total
    
    lr = scheduler.get_last_lr()[0] if scheduler else lr_initial
    print(f"Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy*100:.4f}% at LR: {lr:.6f}")
    
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_accuracy, epoch)
    
    # Validation phase
    if (epoch + 1) % validation_interval == 0:
        model.eval()  
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        val_ood_running_loss = 0.0
        val_ood_correct = 0
        val_ood_total = 0
        
        with torch.no_grad():  
            
            # validation
            for val_images, val_labels in val_dl:
                val_images, val_labels = val_images.to(device), val_labels.to(device)
                
                val_outputs = model(val_images)
                val_loss = criterion(val_outputs, val_labels)
                
                val_running_loss += val_loss.item() * val_images.size(0)
                _, val_predicted = torch.max(val_outputs.data, 1)
                val_total += val_labels.size(0)
                val_correct += (val_predicted == val_labels).sum().item()
        
        # val loss and acc
        val_loss = val_running_loss / val_total
        val_accuracy = val_correct / val_total

        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy*100:.4f}%")
        
        writer.add_scalar('Loss/validation', val_loss, epoch)
        writer.add_scalar('Accuracy/validation', val_accuracy, epoch)
        
     
     
        # BEST MODEL SELECTION
        # For now, we are saving all promising models
        #   - best losses and accuracies for val and ood sets
        # That will make 4 + 1 models saved at the end of training for each run.
        # The Patience thing is only with the in-domain validation set - and early stopping is disabled for now.
        # Of these, the most promising should be the one with best ood val loss or maybe accuracy - is what i think.
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_loss_epoch = epoch + 1
            patience_counter_loss = 0

            best_loss_model_name = f"best_loss_model_{run_id}.pth"
            best_loss_model_path = os.path.join(models_path, best_loss_model_name)
            
            if os.path.exists(best_loss_model_path):
                os.remove(best_loss_model_path)
            
            save_model_weights(model, best_loss_model_path, verbose=False)
            print(f"Saved new best loss model. Val Loss: {best_val_loss:.4f} at Epoch {best_loss_epoch}")
            writer.add_scalar('Best_Validation_Loss', best_val_loss, epoch)
            
        else:
            patience_counter_loss += 1

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_acc_epoch = epoch + 1
            patience_counter_acc = 0

            best_acc_model_name = f"best_acc_model_{run_id}.pth"
            best_acc_model_path = os.path.join(models_path, best_acc_model_name)
            
            if os.path.exists(best_acc_model_path):
                os.remove(best_acc_model_path)
                
            save_model_weights(model, best_acc_model_path, verbose=False)
            print(f"Saved new best accuracy model. Val Acc: {best_val_accuracy*100:.4f}% at Epoch {best_acc_epoch}")
            writer.add_scalar('Best_Validation_Accuracy', best_val_accuracy, epoch)
            
        else:
            patience_counter_acc += 1
        
        # Early stopping based on patience
        if early_stopping:
            if patience_counter_loss >= patience_loss and patience_counter_acc >= patience_accuracy:
                print(f"\n\nEarly stopping would be triggered at epoch {epoch + 1}.\n\n")
                early_stop_path_name = f"early_stop_model_{run_id}.pth"
                early_stop_model_path = os.path.join(models_path, early_stop_path_name)
                save_model_weights(model, early_stop_model_path, verbose=False)
                break # Uncomment this line to enable early stopping
            
    
    # Update learning rate
    if scheduler:
        scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Learning_Rate', current_lr, epoch)
    
        
print("\nTraining complete.")

print(f"\nBest Validation Accuracy: {best_val_accuracy*100:.4f}% at Epoch {best_acc_epoch}")
print(f"Best Validation Loss: {best_val_loss:.4f} at Epoch {best_loss_epoch}")

print(f"\nFinal Training Loss (at end of last epoch): {train_loss:.4f}")
print(f"Final Training Accuracy (at end of last epoch): {train_accuracy*100:.4f}%")

completed_all_epochs = (epoch + 1 == num_epochs)
end_time = time.time()
elapsed_time = end_time - start_time
elapsed_time_hms = time.strftime("%H:%M:%S", time.gmtime(elapsed_time))
print(f"Total Training Time: {elapsed_time_hms}")
print(f"{timestamp}_{training_run_name}_{timm_model_name}")
  
# Save the final model
final_model_name = f"final_model_{timestamp}_{training_run_name}_{timm_model_name}.pth"
final_model_file_path = os.path.join(models_path, final_model_name)
save_model_weights(model, final_model_file_path)

Starting training for run: 09-03_10-30--55_g1_pot_convnextv2_nano.fcmae_ft_in22k_in1k


Epoch 1/50: 100%|██████████| 21/21 [00:09<00:00,  2.17batch/s]

Training Loss: 1.9401, Training Accuracy: 31.3253% at LR: 0.000500


Validation Loss: 2.2350, Validation Accuracy: 18.1818%
Saved new best loss model. Val Loss: 2.2350 at Epoch 1
Saved new best accuracy model. Val Acc: 18.1818% at Epoch 1


Epoch 2/50: 100%|██████████| 21/21 [00:10<00:00,  1.94batch/s]

...

Training Loss: 1.7781, Training Accuracy: 33.7349% at LR: 0.000022


Validation Loss: 1.8736, Validation Accuracy: 27.2727%


Epoch 50/50: 100%|██████████| 21/21 [00:07<00:00,  2.69batch/s]

Training Loss: 1.7773, Training Accuracy: 33.7349% at LR: 0.000020


Validation Loss: 1.8734, Validation Accuracy: 27.2727%

Training complete.

Best Validation Accuracy: 27.2727% at Epoch 3
Best Validation Loss: 1.8624 at Epoch 27

Final Training Loss (at end of last epoch): 1.7773
Final Training Accuracy (at end of last epoch): 33.7349%
Total Training Time: 00:08:18
09-03_10-30--55_g1_pot_convnextv2_nano.fcmae_ft_in22k_in1k
